# Food Recognition & 3D Plate Scanning Research Pipeline
### Kaggle Cloud GPU Training: 43-Class Authentic Indian Food Recognition (15,404 Images)

**Source of Truth Repository:** `https://github.com/VaibhavBajpaij/food-plat-scanner-ai-model.git`  
**Model Architecture:** MobileNetV3-Small (Pretrained Transfer Learning & Staged Bottleneck Fine-Tuning)  
**Dataset:** Enhanced Indian Food Classification (15,404 images across 43 regional Indian classes)  

--- 
### ⚠️ Critical Pipeline & Compliance Rules
1. **Strict 43-Class / 15,404-Image Target:** This notebook explicitly consumes the full 15,404-image Indian food dataset. It fails loudly if the legacy 172-image 5-class dataset is loaded.
2. **Ephemeral Disk Storage:** `/kaggle/working` is wiped when the notebook session closes. All checkpoints (`mobilenet_v3_indian_43class_best.pth`) and evaluation reports are packaged into `kaggle_indian_43class_artifacts.zip` for download.
3. **Zero Hardcoded Secrets:** Do not hardcode API tokens or credentials in this notebook.
4. **Internet Setting:** Ensure that **Internet Access** is toggled **ON** in the Kaggle settings sidebar.

### Step 1: Clone GitHub Repository (Source of Truth)

In [ ]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/VaibhavBajpaij/food-plat-scanner-ai-model.git"
REPO_DIR = "/kaggle/working/food-plat-scanner-ai-model"

if os.path.exists(REPO_DIR):
    print(f"Repository already exists at {REPO_DIR}. Pulling latest commits...")
    subprocess.run(["git", "pull", "origin", "main"], cwd=REPO_DIR, check=True)
else:
    print(f"Cloning source of truth repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"Current Working Directory: {os.getcwd()}")

### Step 2: Check GPU Accelerator, Outgoing Internet & Available Disk Space

In [ ]:
import shutil
import urllib.request
import torch

print("=" * 85)
print("KAGGLE RUNTIME, GPU, INTERNET & STORAGE AUDIT")
print("=" * 85)

# 1. GPU Check
!nvidia-smi
cuda_available = torch.cuda.is_available()
print(f"\nPyTorch Version:         {torch.__version__}")
print(f"CUDA Available:          {cuda_available}")
if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU Accelerator:  {device_name}")
    print(f"Total Dedicated VRAM:    {total_mem_gb:.2f} GB")
else:
    print("WARNING: GPU accelerator is NOT active! Navigate to Settings -> Accelerator -> GPU T4 x2 or P100.")

# 2. Internet Connectivity Check
try:
    with urllib.request.urlopen("https://huggingface.co", timeout=5) as response:
        print("Internet Connectivity:   ONLINE (Direct outbound HTTP access verified)")
except Exception as e:
    print(f"WARNING: Internet access test failed: {e}")
    print("Please toggle Internet -> ON in the Kaggle right-hand settings panel!")

# 3. Disk Space Check
total_disk, used_disk, free_disk = shutil.disk_usage("/kaggle/working")
print(f"Disk Space (/kaggle/working): Total={total_disk/(1024**3):.1f} GB | Free={free_disk/(1024**3):.1f} GB")
assert free_disk / (1024**3) > 4.0, "Error: Less than 4 GB disk space available!"
print("=" * 85)

### Step 3: Install Required Dependencies (Minimal Footprint)

In [ ]:
print("Installing dependencies for Parquet reading, fast decoding, and image processing...")
!pip install -q pyarrow fastparquet Pillow torchvision
print("Dependencies installed successfully.")

### Step 4: Run Resumable Downloader for Approved Parquet Shards (15,404 Images)
Downloads the 5 Parquet shards directly to `/kaggle/working/datasets/raw/sohlhealth_indian_food/`.
- Supports HTTP Range resumption.
- Verifies file integrity and sizes.

In [ ]:
from src.data.resumable_downloader import EnhancedIndianFoodAdapter

adapter = EnhancedIndianFoodAdapter(staging_dir="/kaggle/working/datasets/raw/sohlhealth_indian_food")
plan = adapter.plan()
print(f"Dataset: {plan['name']} | Licence: {plan['licence']}")
print(f"Total Images: {plan['total_images']:,} across {plan['classes_count']} classes | Size: ~{plan['estimated_download_mb']} MB")

success = adapter.acquire(resume=True)
assert success, "Fatal: Parquet shard acquisition failed!"
print("All 5 Parquet shards acquired and verified successfully.")

### Step 5: Parquet Dataset Extraction, Image Decoding & 43-Class Validation
Extracts images from Parquet structs, tests PIL decodability, computes SHA256 and dHash perceptual hashes,
enforces zero cross-split near-duplicate leakage, and generates `indian_food_43class_manifest.json`.

In [ ]:
from src.data.parquet_food_loader import extract_and_prepare_43class_manifest

parquet_dir = "/kaggle/working/datasets/raw/sohlhealth_indian_food"
image_out_dir = "/kaggle/working/datasets/processed/indian_food_43class"
manifest_out_path = "/kaggle/working/data_manifests/indian_food_43class_manifest.json"

manifest_result = extract_and_prepare_43class_manifest(
    parquet_dir=parquet_dir,
    output_image_dir=image_out_dir,
    output_manifest_path=manifest_out_path
)

# Verify counts
assert manifest_result["classes_count"] == 43, f"Expected 43 classes, got {manifest_result['classes_count']}"
assert manifest_result["total_images"] >= 14000, f"Expected >= 14,000 images, got {manifest_result['total_images']}"
print(f"\nSUCCESS: Validated {manifest_result['total_images']:,} authentic Indian food images across {manifest_result['classes_count']} classes!")

### Step 6: Execute Scientific Unit & Integrity Tests

In [ ]:
!python -m src.utils.run_unit_tests

### Step 7: Run 43-Class Pretrained MobileNetV3-Small GPU Smoke Test
Verifies 43-class head initialization, CUDA memory allocation, and forward-backward gradient flow before full training.

In [ ]:
import torch
from src.classification.train_kaggle import build_mobilenet_v3

print("=" * 85)
print("43-CLASS SMOKE TEST: MobileNetV3-Small Initialization & CUDA Gradient Check")
print("=" * 85)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
smoke_model = build_mobilenet_v3(num_classes=43, device=device)

dummy_input = torch.randn(4, 3, 224, 224, device=device)
dummy_target = torch.tensor([0, 15, 27, 42], device=device)
criterion = torch.nn.CrossEntropyLoss()

smoke_model.train()
output = smoke_model(dummy_input)
loss = criterion(output, dummy_target)
loss.backward()

print(f"Smoke Output Shape: {output.shape} (Expected: [4, 43])")
print(f"Smoke Loss:         {loss.item():.4f}")
assert output.shape == (4, 43), "Error: Output shape does not match 43 classes!"
print("SMOKE TEST PASSED: 43-class MobileNetV3-Small pipeline verified on GPU.")
print("=" * 85)

### Step 8: Execute Staged Transfer Learning Training on 15,404 Images (43 Classes)
- **Stage 1**: Custom 43-class classification head on frozen MobileNetV3-Small backbone (5 Epochs).
- **Stage 2**: Staged fine-tuning of upper bottleneck blocks 10-12 with Cosine Annealing (10 Epochs).
- Evaluates against validation and independent test splits with Wilson 95% confidence intervals.
- Saves to distinct checkpoint: `mobilenet_v3_indian_43class_best.pth`.

In [ ]:
from src.classification.train_kaggle import train_indian_43class_model

training_report = train_indian_43class_model(
    manifest_path="/kaggle/working/data_manifests/indian_food_43class_manifest.json",
    epochs_head=5,
    epochs_finetune=10,
    batch_size=32,
    lr_head=1e-3,
    lr_finetune=1e-4,
    seed=42,
    num_workers=2
)

### Step 9: Verify Checkpoint Integrity & Package Artifacts for Recovery
Packages the 43-class checkpoint (`mobilenet_v3_indian_43class_best.pth`), evaluation metrics report,
and manifest into `/kaggle/working/kaggle_indian_43class_artifacts.zip` for download before session closes.

In [ ]:
import json
import shutil
import zipfile
import torch

ckpt_path = "/kaggle/working/models/mobilenet_v3_indian_43class_best.pth"
assert os.path.exists(ckpt_path), f"Trained checkpoint not found at {ckpt_path}!"

size_mb = os.path.getsize(ckpt_path) / (1024**2)
print(f"Verified 43-Class Checkpoint: {ckpt_path} ({size_mb:.2f} MB)")

# Check reloadability
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
ckpt_data = torch.load(ckpt_path, map_location=device, weights_only=False)
assert ckpt_data.get("num_classes") == 43, f"Expected 43 classes in checkpoint, got {ckpt_data.get('num_classes')}"
print(f"Checkpoint Validation: PASS (Epoch {ckpt_data.get('epoch')}, Best Val Acc: {ckpt_data.get('val_acc', 0.0):.1%})")

# Package artifacts for download
zip_path = "/kaggle/working/kaggle_indian_43class_artifacts.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(ckpt_path, arcname="mobilenet_v3_indian_43class_best.pth")
    rep_path = "/kaggle/working/outputs/kaggle_indian_43class_training_report.json"
    if os.path.exists(rep_path):
        zipf.write(rep_path, arcname="kaggle_indian_43class_training_report.json")
    man_path = "/kaggle/working/data_manifests/indian_food_43class_manifest.json"
    if os.path.exists(man_path):
        zipf.write(man_path, arcname="indian_food_43class_manifest.json")

zip_size_mb = os.path.getsize(zip_path) / (1024**2)
print(f"\nArtifact Package Created: {zip_path} ({zip_size_mb:.2f} MB)")
print("=" * 85)
print("ACTION REQUIRED: Download 'kaggle_indian_43class_artifacts.zip' from Kaggle Output tab before session closes!")
print("=" * 85)